# Install dependencies

In [1]:
pip install transformers datasets scikit-learn torch


^C
Note: you may need to restart the kernel to use updated packages.


# Imports

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, AdamW
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


# Load your CSV files

In [ ]:
DATA_DIR = "E:/multimodal_pipeline/data"

train_df = pd.read_csv(os.path.join(DATA_DIR, "fakeddit_train_metadata.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "fakeddit_validate_metadata.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "fakeddit_test_metadata.csv"))


# Dataset Class

In [ ]:
class TextOnlyDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=96):
        self.texts = df["text"].astype(str).tolist()
        self.labels = df["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx])
        }


# Tokenizer and DataLoaders

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_dataset = TextOnlyDataset(train_df, tokenizer)
val_dataset   = TextOnlyDataset(val_df, tokenizer)
test_dataset  = TextOnlyDataset(test_df, tokenizer)

BATCH_SIZE = 8   # CPU safe

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)


# Load DistilBERT model (CPU-friendly)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6
).to(device)


# Optimizer

In [ ]:
optimizer = AdamW(model.parameters(), lr=3e-5)
EPOCHS = 2               # CPU safe


# Training and Evaluation loops

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        output = model(**batch)
        loss = output.loss
        logits = output.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == batch["labels"]).sum().item()
        total += len(preds)

    return total_loss / len(loader), correct / total


def eval_one_epoch(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            output = model(**batch)

            total_loss += output.loss.item()
            preds = torch.argmax(output.logits, dim=1)

            all_labels.extend(batch["labels"].numpy())
            all_preds.extend(preds.numpy())

            correct += (preds == batch["labels"]).sum().item()
            total += len(preds)

    return total_loss / len(loader), correct / total, all_labels, all_preds


# Train model

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(EPOCHS):
    print(f"\n=== Epoch {epoch+1}/{EPOCHS} ===")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_acc, _, _ = eval_one_epoch(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train → Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"Val   → Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")


# Evaluation on test set

In [ ]:
test_loss, test_acc, labels, preds = eval_one_epoch(model, test_loader)
print("Test Loss:", test_loss)
print("Test Acc:", test_acc)

print("\nClassification Report:")
print(classification_report(labels, preds))

cm = confusion_matrix(labels, preds)
sns.heatmap(cm, annot=False, cmap="Blues")
plt.show()
